# Residual Attention U-Net — Ablation Study

Runs 4 configurations on Kaggle GPU and produces:
- Per-run evaluation outputs (metrics, confusion matrices, predictions, boundary plots)
- Combined `ablation_comparison.csv` with side-by-side results
- Saved model files in `model.keras` for each run

All runs use Cosine Annealing LR schedule.

| Version | Configuration | Input | Loss | Boundary Multiplier |
|:---:|---|---|---|:---:|
| 1 | Proposed (Soft Boundary) | 6-Bands (RGB+NIR+NDVI+NDWI) | Dice + Focal + Boundary-Focal | 0.5 (Soft) |
| 2 | Proposed (Medium Boundary) | 6-Bands (RGB+NIR+NDVI+NDWI) | Dice + Focal + Boundary-Focal | 1.0 (Medium) |
| 3 | Proposed (Strong Boundary) | 6-Bands (RGB+NIR+NDVI+NDWI) | Dice + Focal + Boundary-Focal | 2.0 (Strong) |
| 4 | Proposed (Dual Head) | 6-Bands (RGB+NIR+NDVI+NDWI) | Dice + Focal (Primary Head) | 0.0 (Explicit Boundary Head) |

> **Before running:** Set your GitHub repo URL in the cell below.

In [ ]:
GITHUB_REPO_URL = "https://github.com/ATIK2110018/semantic_segmentation.git"
REPO_DIR = "/kaggle/working/segment"
OUTPUT_BASE = "/kaggle/working/ablation_results"

# Auto-detect dataset directory containing the GeoTIFF files under /kaggle/input/
import os, glob
DATA_PATH = f"{REPO_DIR}/dataset"
if os.path.exists("/kaggle/input"):
    tif_files = glob.glob("/kaggle/input/**/*.tif", recursive=True)
    if tif_files:
        DATA_PATH = os.path.dirname(tif_files[0])
        print(f"Auto-detected dataset path: {DATA_PATH}")
    else:
        print(f"No GeoTIFFs found in /kaggle/input, using default: {DATA_PATH}")
else:
    print(f"Not running in Kaggle environment or no input mounted, using default: {DATA_PATH}")

In [ ]:
import os
os.environ["GIT_TERMINAL_PROMPT"] = "0"  # Prevent git clone from hanging on private repos

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest...")
    res = os.system(f"git -C {REPO_DIR} pull")
    if res != 0:
        print("Pull failed. If your repo is private, please check credentials.")
else:
    print("Cloning repo...")
    res = os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
    if res != 0:
        print("Clone failed. If your repo is private, use: https://<PAT>@github.com/... or upload files directly.")
print("Done.")

In [ ]:
import os
print("Restoring/Upgrading NumPy to ensure binary compatibility...")
os.system("pip install -q --upgrade numpy")
print("Installing rasterio...")
os.system("pip install -q rasterio")
print("Installing other requirements (no-deps mode to preserve version compatibility)...")
os.system(f"pip install -q --no-deps -r {REPO_DIR}/requirements.txt")
print("Dependencies installed successfully.")

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, f"{REPO_DIR}/run_ablation.py",
    "--data_path",        DATA_PATH,
    "--patch_size",       "256",
    "--patch_step",       "128",  # 50% overlap for robust representation learning
    "--epochs",           "200",
    "--batch_size",       "16",
    "--lr",               "1e-4",
    "--patience",         "30",
    "--num_samples",      "5",
    "--seed",             "42",  # Fix seed for 100% deterministic reproducibility
    "--ablation_output",  OUTPUT_BASE,
]

print(f"Command: {' '.join(cmd)}\n")
process = subprocess.run(cmd, cwd=REPO_DIR)
print(f"\nExit code: {process.returncode}")
if process.returncode != 0:
    print("\nERROR: Ablation run failed! Check the output logs above for traceback details.")

## Ablation Comparison Table

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = f"{OUTPUT_BASE}/ablation_comparison.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    display(df)
else:
    print(f"Comparison CSV not found at {csv_path}")

## Training History (All Runs)

In [ ]:
from IPython.display import Image, display, Markdown
import glob

runs = sorted(glob.glob(f"{OUTPUT_BASE}/*/"))
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    history_path = os.path.join(run_dir, "training_history.png")
    if os.path.exists(history_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=history_path, width=800))

## Confusion Matrices

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    cm_path = os.path.join(run_dir, "confusion_matrix.png")
    if os.path.exists(cm_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=cm_path, width=900))

## Per-Class IoU Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    iou_path = os.path.join(run_dir, "per_class_iou.png")
    if os.path.exists(iou_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=iou_path, width=800))

## All Metrics Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    metrics_path = os.path.join(run_dir, "all_metrics_chart.png")
    if os.path.exists(metrics_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=metrics_path, width=900))

## Prediction Samples

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    pred_path = os.path.join(run_dir, "predictions.png")
    if os.path.exists(pred_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=pred_path, width=900))

## Boundary Predictions (Color-Coded)

- 🟢 **Green** = Correct boundary (TP)
- 🔴 **Red** = Missed boundary (FN)
- 🟡 **Yellow** = False boundary (FP)

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    bnd_path = os.path.join(run_dir, "boundary_predictions.png")
    if os.path.exists(bnd_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=bnd_path, width=1000))

## Boundary Metrics Charts

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    chart_path = os.path.join(run_dir, "boundary_metrics_chart.png")
    if os.path.exists(chart_path):
        display(Markdown(f"### {run_name}"))
        display(Image(filename=chart_path, width=900))

## Per-Run Evaluation CSVs

In [ ]:
for run_dir in runs:
    run_name = os.path.basename(run_dir.rstrip('/'))
    eval_csv = os.path.join(run_dir, "evaluation_results.csv")
    bnd_csv = os.path.join(run_dir, "boundary_results_global.csv")
    
    if os.path.exists(eval_csv):
        display(Markdown(f"### {run_name} — Pixel Metrics"))
        display(pd.read_csv(eval_csv))
    if os.path.exists(bnd_csv):
        display(Markdown(f"### {run_name} — Boundary Metrics"))
        display(pd.read_csv(bnd_csv))

## Class Legend

In [ ]:
legend_path = None
for run_dir in runs:
    p = os.path.join(run_dir, "class_legend.png")
    if os.path.exists(p):
        legend_path = p
        break
if legend_path:
    display(Image(filename=legend_path, width=400))

## Download Results

Run the cell below to zip all outputs into a single downloadable file.

In [ ]:
import shutil
from IPython.display import FileLink

shutil.make_archive('/kaggle/working/ablation_results', 'zip', OUTPUT_BASE)
print("Zip created: ablation_results.zip")
display(FileLink('ablation_results.zip'))

In [ ]:
for run_dir in sorted(glob.glob(f"{OUTPUT_BASE}/*/")):
    run_name = os.path.basename(run_dir.rstrip('/'))
    files = os.listdir(run_dir)
    print(f"\n{run_name}/ ({len(files)} files)")
    for f in sorted(files):
        size = os.path.getsize(os.path.join(run_dir, f))
        print(f"  {f:.<45} {size/1024:.1f} KB")

comp = f"{OUTPUT_BASE}/ablation_comparison.csv"
if os.path.exists(comp):
    print(f"\nablation_comparison.csv .... {os.path.getsize(comp)/1024:.1f} KB")